# HMM Market Regime Detection

دفتر تعليمي مختصر يوضح كيف نستعمل Hidden Markov Models لاكتشاف أنظمة السوق اعتمادا على بيانات مالية يومية.

**تنبيه:** هذا notebook لأغراض تعليمية وبحثية فقط، وليس نصيحة مالية.

## الفكرة

نفترض أن السوق ينتقل بين حالات مخفية مثل سوق صاعد، سوق هابط، أو سوق عالي التقلب. لا نرى هذه الحالات مباشرة، لكننا نرى العوائد والتقلب والحجم. نموذج HMM يحاول استنتاج الحالة الأكثر احتمالا لكل يوم.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_market_data
from src.features import build_features
from src.hmm_model import train_hmm, predict_states, attach_states, summarize_states
from src.backtest import run_regime_strategy
from src.visualization import plot_price_regimes, plot_equity_curves


## تحميل البيانات

نستعمل الرمز `^GSPC` لمؤشر S&P 500. يمكن تغييره إلى `AAPL` أو أي رمز مدعوم في Yahoo Finance.

In [ ]:
ticker = '^GSPC'
data = load_market_data(ticker=ticker, start='2015-01-01')
data.head()

## حساب الخصائص المالية

نحسب العائد اليومي، التقلب، ميل المتوسطات المتحركة، وتغير الحجم. بعدها يتم توحيد الخصائص عبر StandardScaler.

In [ ]:
features = build_features(data)
features.frame[features.columns].head()

## تدريب HMM واستخراج الحالات

In [ ]:
model = train_hmm(features.matrix, n_components=3, covariance_type='full', random_state=42)
states = predict_states(model, features.matrix)
regime_frame = attach_states(features.frame, states)
state_summary = summarize_states(regime_frame)
state_summary

## رسم السعر مع الحالات

In [ ]:
figures_dir = PROJECT_ROOT / 'outputs' / 'figures'
plot_price_regimes(regime_frame, figures_dir / 'notebook_hmm_market_regimes.png', title=f'{ticker} HMM Regimes')

## Backtest تعليمي

ندخل Long فقط في الحالات ذات العائد المتوسط الموجب والتقلب غير المرتفع. لا يوجد leverage ولا short.

In [ ]:
backtest = run_regime_strategy(regime_frame, state_summary, transaction_cost=0.0005)
plot_equity_curves(backtest.equity_frame, figures_dir / 'notebook_hmm_backtest_equity.png')
backtest.metrics

## خلاصة تعليمية

- HMM يساعد على تصنيف السوق إلى حالات مخفية قابلة للتفسير.
- جودة التصنيف لا تعني بالضرورة استراتيجية مربحة.
- يجب اختبار النموذج خارج العينة وإضافة إدارة مخاطر قبل أي استعمال عملي.
- هذا التطبيق بداية بحثية قابلة للتطوير وليست نظام تداول جاهز.